# Phase 3: Real Point Cloud Processing & OptiTrack Alignment

This notebook prepares raw physical scan data (`view*.pcd`) for surface analysis and PointNet benchmarking:
1. **RANSAC Plane Segmentation**: Cleans table/ground plane noise from physical scans.
2. **Oriented Bounding Box (OBB) Cropping**: Isolates the workpiece ROI.
3. **OptiTrack Ground Truth Alignment**: Uses the physical OptiTrack pose with manual marker-to-CAD offset ($T_{base \to CAD} = T_{optitrack} \cdot T_{offset}$).
4. **Export for Surface Analysis**: Saves cleaned scans into `processed_data/` so `6_surface_analysis.ipynb` can compute per-surface Chamfer Distances.

## 1. Helper Functions

In [1]:
import os
import re
import copy
import glob
import math
import numpy as np
import open3d as o3d
from scipy.spatial.transform import Rotation as R
import pandas as pd

def get_rotation_matrix_z(deg):
# Creates a 3x3 rotation matrix for the Z-axis.
    rad = math.radians(deg)
    c, s = math.cos(rad), math.sin(rad)
    return np.array([[c, -s, 0],
                     [s,  c, 0],
                     [0,  0, 1]])

def pose_to_matrix(pose):
    """
    Converts [x, y, z, roll, pitch, yaw] (angles in degrees) to a 4x4 transformation matrix.
    """
    x, y, z = pose[:3]
    roll, pitch, yaw = pose[3:]
    rot_matrix = R.from_euler('xyz', [roll, pitch, yaw], degrees=True).as_matrix()
    T = np.eye(4)
    T[:3, :3] = rot_matrix
    T[:3, 3] = [x, y, z]
    return T

def clean_and_crop_point_cloud(pcd, initial_pos, initial_angle, box_size, remove_plane=True, distance_threshold=3.0, plane_offset=0.5):
    """
    Processes a single raw physical PCD: removes ground plane and crops to OBB.
    """
    if pcd.is_empty():
        return pcd
    
    pcd_clean = copy.deepcopy(pcd)
    
    # 1. Detect and remove table/ground plane using RANSAC
    if remove_plane:
        plane_model, inliers = pcd_clean.segment_plane(distance_threshold=distance_threshold, ransac_n=3, num_iterations=2000)
        [a, b, c, d] = plane_model
        pts = np.asarray(pcd_clean.points)
        distances = a * pts[:, 0] + b * pts[:, 1] + c * pts[:, 2] + d
        above_plane_indices = np.where(distances > plane_offset)[0]
        pcd_clean = pcd_clean.select_by_index(above_plane_indices)
        
    # 2. Crop to Oriented Bounding Box around the workpiece
    rot_matrix = get_rotation_matrix_z(-initial_angle)
    obb = o3d.geometry.OrientedBoundingBox(
        center=np.array(initial_pos),
        R=rot_matrix,
        extent=np.array(box_size)
    )
    final_pcd = pcd_clean.crop(obb)
    return final_pcd

def extract_number(filename):
# Extracts numerical ID from filename (e.g. view12.pcd -> 12).
    numbers = re.findall(r'\d+', os.path.basename(filename))
    return int(numbers[0]) if numbers else 0


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## 2. Interactive Visual Validation & Manual Offset Adjustment
Set your OptiTrack matrix and manual offsets below ($T_{base \to CAD} = T_{optitrack} \cdot T_{offset}$).

- `OFFSET_TRANSLATION_MM`: `[dx, dy, dz]` offset in mm from OptiTrack marker pivot to CAD origin.
- `OFFSET_ROTATION_DEG`: `[roll, pitch, yaw]` rotation adjustment in degrees.

In [7]:
# ==========================================
# 1. CONFIGURATION: EXPERIMENT & WORKPIECE
# ==========================================
EXPERIMENT = "test_11_real_pocket"
WORKPIECE = "TH1021AV"
VIEWPOINT_IDX = 231

DATA_DIR = f"pcd_data/testing_data/{EXPERIMENT}"
if os.path.exists(os.path.join(DATA_DIR, WORKPIECE)):
    DATA_DIR = os.path.join(DATA_DIR, WORKPIECE)
CAD_PATH = f"workpiece/{WORKPIECE}/workpiece.stl"

# Load Initial YOLO / Rough Object Pose for Cropping Box
yolo_pose_path = os.path.join(DATA_DIR, "initial_obj_pose.npy")
if os.path.exists(yolo_pose_path):
    tf_obj = np.load(yolo_pose_path)
    YOLO_POS = tf_obj[:3].copy()
    YOLO_ANGLE = float(tf_obj[4])
    YOLO_POS[0] -= 0  # Visual crop adjustment if needed
    YOLO_POS[1] -= 0
else:
    raise ValueError("No Yolo value found")

# Define Bounding Box Size (mm) around workpiece: [X_extent, Y_extent, Z_extent]
# CROP_BOX = [80, 105, 70]  # Adjust ROI dimensions for your workpiece
CROP_BOX = [80, 60, 70]
# ==========================================
# 2. LOAD OPTITRACK GROUND TRUTH
# ==========================================
optitrack_path = os.path.join(DATA_DIR, "T_average.npy")
if not os.path.exists(optitrack_path):
    print("Using Optitrack Ground Truth")
    optitrack_path = os.path.join(DATA_DIR, "T_optitrack.npy")

if os.path.exists(optitrack_path):
    T_gt_raw = np.load(optitrack_path)
    T_gt_raw[3, :] = [0, 0, 0, 1]
    GT_ROTATION_Z_DEG = 0.0 if "T_average" in optitrack_path else 90.0
    rad_z = math.radians(GT_ROTATION_Z_DEG)
    T_local_rot = np.eye(4)
    T_local_rot[:3, :3] = np.array([
        [math.cos(rad_z), -math.sin(rad_z), 0],
        [math.sin(rad_z),  math.cos(rad_z), 0],
        [0, 0, 1]
    ])
    T_ground_truth = T_gt_raw @ T_local_rot
    T_ground_truth[3, :] = [0, 0, 0, 1]
else:
    raise FileNotFoundError(f"Missing Ground Truth OptiTrack file at {optitrack_path}")

print("Final Ground Truth Transformation Matrix (T_ground_truth):")
print(T_ground_truth)
print(f"  - Position (mm): {np.round(T_ground_truth[:3, 3], 2)}")

# ==========================================
# 4. LOAD CAD & REAL SCAN (SINGLE VIEW)
# ==========================================
mesh = o3d.io.read_triangle_mesh(CAD_PATH)
mesh.compute_vertex_normals()
cad_cloud = mesh.sample_points_uniformly(number_of_points=40000)

pcd_file = os.path.join(DATA_DIR, f"view{VIEWPOINT_IDX}.pcd")
if not os.path.exists(pcd_file):
    pcd_file = os.path.join(DATA_DIR, f"view{VIEWPOINT_IDX:02d}.pcd")

raw_pcd = o3d.io.read_point_cloud(pcd_file)

# Create Oriented Bounding Box for Visualization
rot_matrix = get_rotation_matrix_z(-YOLO_ANGLE)
obb = o3d.geometry.OrientedBoundingBox(
    center=np.array(YOLO_POS),
    R=rot_matrix,
    extent=np.array(CROP_BOX)
)
obb.color = (0.0, 1.0, 0.0)  # Green wireframe box

# Clean table plane and crop to OBB
clean_pcd = clean_and_crop_point_cloud(raw_pcd, YOLO_POS, YOLO_ANGLE, CROP_BOX, remove_plane=True)

# ==========================================
# 5. VISUAL VERIFICATION & CROP BOX INSPECTION
# ==========================================
cad_aligned = copy.deepcopy(cad_cloud).transform(T_ground_truth)
cad_aligned.paint_uniform_color([1.0, 0.0, 0.0])       # Red: CAD Ground Truth
clean_pcd.paint_uniform_color([0.0, 0.65, 0.93])      # Blue: Real Scanned Point Cloud
raw_pcd_vis = copy.deepcopy(raw_pcd).paint_uniform_color([0.6, 0.6, 0.6]) # Grey: Raw Uncropped

world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=50.0, origin=T_ground_truth[:3, 3])
crop_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=30.0, origin=YOLO_POS)

print(f"\n1. Window 1: RAW Scan (Grey) + GREEN Cropping Box...")
print(f"   -> Verify the green box fits around the workpiece before closing window.")
o3d.visualization.draw_geometries([raw_pcd_vis, obb], window_name="1. Raw Scan & Cropping Box (Green Wireframe)")

print(f"\n2. Window 2: CROPPED Scan (Blue) + CAD Model (Red) + Cropping Box (Green)...")
o3d.visualization.draw_geometries([clean_pcd, cad_aligned, obb, world_frame], window_name=f"2. OptiTrack Alignment & Crop - View {VIEWPOINT_IDX}")


Using Optitrack Ground Truth
Final Ground Truth Transformation Matrix (T_ground_truth):
[[ 9.36165607e-03 -9.99956179e-01  0.00000000e+00  5.60059260e+02]
 [ 9.99956179e-01  9.36165607e-03  0.00000000e+00 -9.03420747e+01]
 [ 0.00000000e+00  0.00000000e+00  1.00000000e+00 -1.03167702e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]
  - Position (mm): [560.06 -90.34  -1.03]

1. Window 1: RAW Scan (Grey) + GREEN Cropping Box...
   -> Verify the green box fits around the workpiece before closing window.

2. Window 2: CROPPED Scan (Blue) + CAD Model (Red) + Cropping Box (Green)...


In [4]:
# o3d.visualization.draw_geometries([cad_aligned, world_frame])

# T_ground_truth = T_gt_raw @ T_local_rot
# yolo_pose_path = os.path.join(DATA_DIR, "initial_obj_pose.npy")
# if os.path.exists(yolo_pose_path):


tf_obj = np.load("pcd_data/testing_data/test_11_real_pocket\TH1021AV\initial_obj_pose.npy")
print(tf_obj)
tf_opt = np.load("pcd_data/testing_data/test_11_real_pocket\TH1021AV\T_optitrack.npy")
print(tf_opt,"\n")

# tf_obj = np.load("pcd_data/testing_data/test_11_real_pocket\TH1031AV\initial_obj_pose.npy")
# print(tf_obj)
# tf_opt = np.load("pcd_data/testing_data/test_11_real_pocket\TH1031AV\T_optitrack.npy")
# print(tf_opt,"\n")

# tf_obj = np.load("pcd_data/testing_data/test_11_real_pocket\TH1041AV\initial_obj_pose.npy")
# print(tf_obj)
# tf_opt = np.load("pcd_data/testing_data/test_11_real_pocket\TH1041AV\T_optitrack.npy")
# print(tf_opt,"\n")


[558.06223279 -87.69807311  25.92737606   0.           2.06999993
   0.        ]
[[ 9.99904245e-01  9.45273354e-03  1.01067794e-02  5.59553114e+02]
 [-9.36116986e-03  9.99915058e-01 -9.06887585e-03 -8.98854876e+01]
 [-1.01916466e-02  8.97339618e-03  9.99907800e-01 -7.94247261e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]] 



In [3]:
# 1. Origin 0,0,0 (Robot Base Coordinate System)
origin_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=150.0, origin=[0, 0, 0])

# 2. Scanned/actual workpiece
# We will show the raw uncropped scan in grey so you can see ALL objects on the table
raw_pcd_vis = copy.deepcopy(raw_pcd).paint_uniform_color([0.6, 0.6, 0.6])

# 3. The box that it creates using obb (Green Box generated by YOLO)
# This was defined as 'obb' in the previous cell
obb.color = (0.0, 1.0, 0.0)

# 4. The frame of OptiTrack (Where Ground Truth says the workpiece is)
opti_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=80.0, origin=T_ground_truth[:3, 3])

print("\\nVisualizing: Origin (0,0,0), Entire Raw Scan, YOLO Box (Green), and OptiTrack Frame...")
o3d.visualization.draw_geometries([origin_frame, raw_pcd_vis, obb, opti_frame], window_name="Debugging YOLO vs OptiTrack Mismatch")


\nVisualizing: Origin (0,0,0), Entire Raw Scan, YOLO Box (Green), and OptiTrack Frame...


## 3. Batch Processing Loop (Exporting for Surface Analysis)
This cell cleans and processes all real viewpoint scans, transforms them to the CAD coordinate frame using the manual OptiTrack ground truth pose, and exports them directly into `processed_data/` so that **`6_surface_analysis.ipynb`** can segment the surface features and generate `metadata.csv`.

In [10]:
# ==========================================
# BATCH REAL DATA PROCESSING PIPELINE
# ==========================================
EXPERIMENT = "test_11_real_pocket"
WORKPIECES = ["TH1021AV", "TH1031AV", "TH1041AV", "TH1051AV"]

for workpiece in WORKPIECES:
    print(f"\n========================================")
    print(f"PROCESSING REAL WORKPIECE: {workpiece}")
    print(f"========================================")
    
    DATA_DIR = f"pcd_data/testing_data/{EXPERIMENT}"
    if os.path.exists(os.path.join(DATA_DIR, workpiece)):
        DATA_DIR = os.path.join(DATA_DIR, workpiece)
        
    CAD_PATH = f"workpiece/{workpiece}/workpiece.stl"
    PROCESSED_DIR = f"processed_data/{EXPERIMENT}/{workpiece}"
    EVAL_DIR = f"evaluation_result/{EXPERIMENT}/{workpiece}"
    os.makedirs(PROCESSED_DIR, exist_ok=True)
    os.makedirs(EVAL_DIR, exist_ok=True)
    
    # 1. Compute and Save Ground Truth Matrix
    optitrack_path = os.path.join(DATA_DIR, "T_average.npy")
    if not os.path.exists(optitrack_path):
        optitrack_path = os.path.join(DATA_DIR, "T_optitrack.npy")
    if os.path.exists(optitrack_path):
        T_gt_raw = np.load(optitrack_path)
        T_gt_raw[3, :] = [0, 0, 0, 1]
        GT_ROTATION_Z_DEG = 0.0 if "T_average" in optitrack_path else 90.0
        rad_z = math.radians(GT_ROTATION_Z_DEG)
        T_local_rot = np.eye(4)
        T_local_rot[:3, :3] = np.array([
            [math.cos(rad_z), -math.sin(rad_z), 0],
            [math.sin(rad_z),  math.cos(rad_z), 0],
            [0, 0, 1]
        ])
        T_gt = T_gt_raw @ T_local_rot
        T_gt[3, :] = [0, 0, 0, 1]
    else:
        raise FileNotFoundError(f"Missing Ground Truth OptiTrack file at {optitrack_path}")

    yolo_pose_path = os.path.join(DATA_DIR, "initial_obj_pose.npy")
    if os.path.exists(yolo_pose_path):
        tf_obj = np.load(yolo_pose_path)
        YOLO_POS = tf_obj[:3].copy()
        YOLO_ANGLE = float(tf_obj[4])
        YOLO_POS[1] -= -10.0
    else:
        raise FileNotFoundError(f"Missing Ground Truth OptiTrack file at {optitrack_path}")
    
    # 2. Find all real scan files (e.g. view*.pcd)
    pcd_files = [f for f in os.listdir(DATA_DIR) if f.startswith("view") and f.endswith(".pcd") and "_surface" not in f]
    pcd_files.sort(key=extract_number)
    
    print(f"Found {len(pcd_files)} real scan files. Processing and aligning to CAD frame...")
    
    T_inv_gt = np.linalg.inv(T_gt)  # Transform real points into CAD coordinate frame
    
    exported_count = 0
    for file_name in pcd_files:
        view_idx = extract_number(file_name)
        file_path = os.path.join(DATA_DIR, file_name)
        raw_pcd = o3d.io.read_point_cloud(file_path)
        
        # Clean ground plane & Crop workpiece ROI
        clean_pcd = clean_and_crop_point_cloud(raw_pcd, YOLO_POS, YOLO_ANGLE, CROP_BOX, remove_plane=True)
        
        if len(clean_pcd.points) < 50:
            print(f"  [Warning] Viewpoint {view_idx} has too few points ({len(clean_pcd.points)}), skipping.")
            continue
            
        # Transform scan directly into CAD coordinate frame for 6_surface_analysis
        clean_pcd.transform(T_inv_gt)
        
        # Export with standard naming format recognized by 6_surface_analysis.ipynb
        export_filename = f"viewpoint_simulated_noise_{view_idx}.pcd"
        export_path = os.path.join(PROCESSED_DIR, export_filename)
        o3d.io.write_point_cloud(export_path, clean_pcd)
        exported_count += 1
        
    print(f"[SUCCESS] Exported {exported_count} cleaned & CAD-aligned PCDs to: {PROCESSED_DIR}")

print("\n==========================================")
print("ALL REAL DATA PROCESSED!")
print("You can now run 6_surface_analysis.ipynb to extract surface features & compute Chamfer errors.")
print("==========================================")



PROCESSING REAL WORKPIECE: workpiece31
Saved Ground Truth transformation to: evaluation_result/test_10_realgrasp/workpiece31/merge_full_transformation.npy
Found 287 real scan files. Processing and aligning to CAD frame...
[SUCCESS] Exported 287 cleaned & CAD-aligned PCDs to: processed_data/test_10_realgrasp/workpiece31

ALL REAL DATA PROCESSED!
You can now run 6_surface_analysis.ipynb to extract surface features & compute Chamfer errors.


In [1]:
print("a")

a
